In [5]:

# ── Cell 1: Install & authenticate ───────────────────────────────────────────
# Run this cell once. After kaggle.json is set up you never need to again.
# !pip install kaggle kagglehub segmentation-models-pytorch nibabel opencv-python scikit-learn

import os

# Option A: kaggle.json already in ~/.kaggle/ → nothing to do
# Option B: paste your credentials directly (fine for local dev, never commit this)
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY']      = 'your_api_key'


# ── Cell 2: Download dataset from Kaggle ─────────────────────────────────────
import kagglehub

print("Downloading BraTS2020 dataset from Kaggle...")
KAGGLE_PATH = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
print(f"Downloaded to: {KAGGLE_PATH}")

# Walk one level to find the actual training data folder
# (kagglehub nests files under a version subfolder)
TRAIN_DATASET_PATH = None
for root, dirs, files in os.walk(KAGGLE_PATH):
    if "MICCAI_BraTS2020_TrainingData" in dirs:
        TRAIN_DATASET_PATH = os.path.join(root, "MICCAI_BraTS2020_TrainingData")
        break

# Fallback: try common direct paths
if TRAIN_DATASET_PATH is None:
    candidates = [
        os.path.join(KAGGLE_PATH, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData"),
        os.path.join(KAGGLE_PATH, "MICCAI_BraTS2020_TrainingData"),
        KAGGLE_PATH,
    ]
    for c in candidates:
        if os.path.exists(c):
            TRAIN_DATASET_PATH = c
            break

print(f"Training data path: {TRAIN_DATASET_PATH}")
assert TRAIN_DATASET_PATH is not None, "Could not locate MICCAI_BraTS2020_TrainingData. Print KAGGLE_PATH and check the folder structure manually."

Downloaded to: /Users/yuzheli/.cache/kagglehub/datasets/awsaf49/brats20-dataset-training-validation/versions/1
Training data path: /Users/yuzheli/.cache/kagglehub/datasets/awsaf49/brats20-dataset-training-validation/versions/1/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData


In [6]:
import torch
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import cv2
import numpy as np
import os
from sklearn.model_selection import train_test_split

In [7]:
# TRAIN_DATASET_PATH = "/Users/sherwinvahidimowlavi/Downloads/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/"
IMG_SIZE = 128
VOLUME_SLICES = 100
VOLUME_START_AT = 22
BATCH_SIZE = 8
NUM_EPOCHS = 35
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: mps


In [8]:
import os
from sklearn.model_selection import train_test_split

# Fix the BraTS20_Training_355 file naming issue
old_name = os.path.join(TRAIN_DATASET_PATH, "BraTS20_Training_355/W39_1998.09.19_Segm.nii")
new_name = os.path.join(TRAIN_DATASET_PATH, "BraTS20_Training_355/BraTS20_Training_355_seg.nii")

try:
    if os.path.exists(old_name):
        os.rename(old_name, new_name)
        print("✓ File has been renamed successfully!")
    else:
        print("✓ File already renamed or doesn't exist")
except Exception as e:
    print(f"Rename failed: {e}")

# Get all patient directories
train_and_val_directories = [f.path for f in os.scandir(TRAIN_DATASET_PATH) if f.is_dir()]

def pathListIntoIds(dirList):
    x = []
    for i in range(len(dirList)):
        x.append(dirList[i][dirList[i].rfind('/')+1:])
    return x

# Get patient IDs
train_and_test_ids = pathListIntoIds(train_and_val_directories)

# Split data (70% train, 20% val, 10% test)
train_test_ids, val_ids = train_test_split(train_and_test_ids, test_size=0.2, random_state=42)
train_ids, test_ids = train_test_split(train_test_ids, test_size=0.125, random_state=42)  # 0.125 * 0.8 = 0.1

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

✓ File has been renamed successfully!
Train: 258, Val: 74, Test: 37


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import cv2
import numpy as np

IMG_SIZE = 128
VOLUME_SLICES = 100
VOLUME_START_AT = 22

class BraTSDataset(Dataset):
    def __init__(self, patient_ids, data_path):
        self.patient_ids = patient_ids
        self.data_path = data_path
        
    def __len__(self):
        return len(self.patient_ids) * VOLUME_SLICES
    
    def __getitem__(self, idx):
        # Determine patient and slice
        patient_idx = idx // VOLUME_SLICES
        slice_idx = idx % VOLUME_SLICES
        
        patient_id = self.patient_ids[patient_idx]
        patient_path = os.path.join(self.data_path, patient_id)
        
        # Load MRI data
        flair = nib.load(os.path.join(patient_path, f'{patient_id}_flair.nii')).get_fdata()
        t1ce = nib.load(os.path.join(patient_path, f'{patient_id}_t1ce.nii')).get_fdata()
        seg = nib.load(os.path.join(patient_path, f'{patient_id}_seg.nii')).get_fdata()
        
        # Extract and resize slice
        slice_num = slice_idx + VOLUME_START_AT
        flair_slice = cv2.resize(flair[:, :, slice_num], (IMG_SIZE, IMG_SIZE))
        t1ce_slice = cv2.resize(t1ce[:, :, slice_num], (IMG_SIZE, IMG_SIZE))
        seg_slice = cv2.resize(seg[:, :, slice_num], (IMG_SIZE, IMG_SIZE), 
                              interpolation=cv2.INTER_NEAREST)
        
        # Stack channels: (2, H, W) format for PyTorch
        image = np.stack([flair_slice, t1ce_slice], axis=0)
        
        # Fix labels: 4 -> 3
        seg_slice[seg_slice == 4] = 3
        
        # Normalize
        image = image / np.max(image) if np.max(image) > 0 else image
        
        # Convert to tensors
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(seg_slice).long()
        
        return {'image': image, 'mask': mask}

In [10]:
BATCH_SIZE = 8

# Create datasets
train_dataset = BraTSDataset(train_ids, TRAIN_DATASET_PATH)
val_dataset = BraTSDataset(val_ids, TRAIN_DATASET_PATH)
test_dataset = BraTSDataset(test_ids, TRAIN_DATASET_PATH)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")
print(f"✓ Test batches: {len(test_loader)}")

# Test loading one batch
sample_batch = next(iter(train_loader))
print(f"✓ Image batch shape: {sample_batch['image'].shape}")  # Should be (8, 2, 128, 128)
print(f"✓ Mask batch shape: {sample_batch['mask'].shape}")    # Should be (8, 128, 128)

✓ Train batches: 3225
✓ Val batches: 925
✓ Test batches: 463
✓ Image batch shape: torch.Size([8, 2, 128, 128])
✓ Mask batch shape: torch.Size([8, 128, 128])


In [ ]:
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Model
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=2,  # FLAIR + T1CE
    classes=4,       
    activation=None  
).to(DEVICE)

# Loss and optimizer
loss_fn = smp.losses.DiceLoss(mode='multiclass')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
NUM_EPOCHS = 35

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    for batch_idx, batch in enumerate(train_loader):
        images = batch['image'].to(DEVICE)
        masks = batch['mask'].to(DEVICE)
        
        # Forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Print progress
        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Batch [{batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(DEVICE)
            masks = batch['mask'].to(DEVICE)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            val_loss += loss.item()
    
    # Epoch summary
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"{'='*60}\n")
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
        }, f'checkpoint_epoch_{epoch+1}.pth')
        print(f"✓ Saved checkpoint: checkpoint_epoch_{epoch+1}.pth\n")

Using device: mps


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Epoch [1/35] Batch [0/3225] Loss: 0.9328
Epoch [1/35] Batch [50/3225] Loss: 0.6771
Epoch [1/35] Batch [100/3225] Loss: 0.5308
Epoch [1/35] Batch [150/3225] Loss: 0.4212
Epoch [1/35] Batch [200/3225] Loss: 0.3860
Epoch [1/35] Batch [250/3225] Loss: 0.4588
Epoch [1/35] Batch [300/3225] Loss: 0.5791
Epoch [1/35] Batch [350/3225] Loss: 0.4832
Epoch [1/35] Batch [400/3225] Loss: 0.3041
Epoch [1/35] Batch [450/3225] Loss: 0.3379
Epoch [1/35] Batch [500/3225] Loss: 0.6214
Epoch [1/35] Batch [550/3225] Loss: 0.5398
Epoch [1/35] Batch [600/3225] Loss: 0.3639
Epoch [1/35] Batch [650/3225] Loss: 0.2998
Epoch [1/35] Batch [700/3225] Loss: 0.2624
Epoch [1/35] Batch [750/3225] Loss: 0.4667
Epoch [1/35] Batch [800/3225] Loss: 0.2995
Epoch [1/35] Batch [850/3225] Loss: 0.3041
Epoch [1/35] Batch [900/3225] Loss: 0.5544
Epoch [1/35] Batch [950/3225] Loss: 0.3483
Epoch [1/35] Batch [1000/3225] Loss: 0.1849
Epoch [1/35] Batch [1050/3225] Loss: 0.3181
Epoch [1/35] Batch [1100/3225] Loss: 0.2812
Epoch [1/35

In [ ]:
import torch
import numpy as np
from scipy.spatial.distance import directed_hausdorff

# ── Dice per class ──────────────────────────────────────────────
def dice_score(pred, target, num_classes=4, smooth=1e-6):
    """
    pred:   (N, H, W) long tensor – predicted class indices
    target: (N, H, W) long tensor – ground truth class indices
    Returns dict with per-class Dice and mean tumor Dice (classes 1-3).
    """
    scores = {}
    for c in range(num_classes):
        p = (pred == c).float()
        t = (target == c).float()
        intersection = (p * t).sum()
        scores[f'dice_class_{c}'] = (
            (2 * intersection + smooth) / (p.sum() + t.sum() + smooth)
        ).item()
    # BraTS convention: mean over tumor classes only (1=NCR, 2=ED, 3=ET)
    scores['mean_tumor_dice'] = np.mean([scores[f'dice_class_{c}'] for c in range(1, 4)])
    return scores


# ── HD95 ────────────────────────────────────────────────────────
def hd95(pred_mask, target_mask):
    """Binary masks (H, W) as numpy arrays. Returns HD95 in pixels."""
    pred_pts   = np.argwhere(pred_mask)
    target_pts = np.argwhere(target_mask)
    if len(pred_pts) == 0 and len(target_pts) == 0:
        return 0.0
    if len(pred_pts) == 0 or len(target_pts) == 0:
        return np.inf
    d1 = directed_hausdorff(pred_pts, target_pts)[0]
    d2 = directed_hausdorff(target_pts, pred_pts)[0]
    # Compute all distances for 95th percentile
    from scipy.spatial import cKDTree
    tree1 = cKDTree(pred_pts)
    tree2 = cKDTree(target_pts)
    d_fwd = tree2.query(pred_pts)[0]
    d_bwd = tree1.query(target_pts)[0]
    all_d = np.concatenate([d_fwd, d_bwd])
    return float(np.percentile(all_d, 95))


def hd95_multiclass(pred, target, num_classes=4):
    """Averages HD95 over tumor classes (1-3)."""
    scores = {}
    for c in range(1, num_classes):
        p = (pred == c).cpu().numpy()
        t = (target == c).cpu().numpy()
        # Average over batch
        batch_hd = [hd95(p[i], t[i]) for i in range(p.shape[0])]
        finite = [x for x in batch_hd if np.isfinite(x)]
        scores[f'hd95_class_{c}'] = np.mean(finite) if finite else np.inf
    scores['mean_tumor_hd95'] = np.mean([scores[f'hd95_class_{c}'] for c in range(1, 4)])
    return scores


# ── Full evaluation loop ────────────────────────────────────────
def evaluate(model, loader, device, num_classes=4):
    model.eval()
    all_dice   = {f'dice_class_{c}': [] for c in range(num_classes)}
    all_dice['mean_tumor_dice'] = []
    all_hd95   = {f'hd95_class_{c}': [] for c in range(1, num_classes)}
    all_hd95['mean_tumor_hd95'] = []

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            masks  = batch['mask'].to(device)

            logits = model(images)                        # (N, C, H, W)
            preds  = torch.argmax(logits, dim=1)          # (N, H, W)

            d = dice_score(preds, masks, num_classes)
            for k, v in d.items():
                all_dice[k].append(v)

            h = hd95_multiclass(preds, masks, num_classes)
            for k, v in h.items():
                all_hd95[k].append(v)

    results = {}
    for k, v in all_dice.items():
        results[k] = np.mean(v)
    for k, v in all_hd95.items():
        results[k] = np.nanmean(v)
    return results


# ── Run & print ─────────────────────────────────────────────────
print("Evaluating on test set...")
test_results = evaluate(model, test_loader, DEVICE)

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"  Dice  Background : {test_results['dice_class_0']:.4f}")
print(f"  Dice  NCR (1)    : {test_results['dice_class_1']:.4f}")
print(f"  Dice  Edema (2)  : {test_results['dice_class_2']:.4f}")
print(f"  Dice  ET  (3)    : {test_results['dice_class_3']:.4f}")
print(f"  Mean Tumor Dice  : {test_results['mean_tumor_dice']:.4f}  ← key metric")
print(f"  HD95  NCR (1)    : {test_results['hd95_class_1']:.2f} px")
print(f"  HD95  Edema (2)  : {test_results['hd95_class_2']:.2f} px")
print(f"  HD95  ET  (3)    : {test_results['hd95_class_3']:.2f} px")
print(f"  Mean Tumor HD95  : {test_results['mean_tumor_hd95']:.2f} px")
print("="*50)

In [ ]:
"""
few_shot_sampler.py
-------------------
Episode-based data loading for k-shot brain tumor segmentation.

Usage:
    sampler = FewShotSampler(support_ids, query_ids, TRAIN_DATASET_PATH, k_shot=5)
    support_batch, query_batch = sampler.sample_episode()
"""

import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import cv2

# Import dice_score — works both as standalone file and inside notebook
try:
    from metrics import dice_score
except ImportError:
    pass  # when running inside the notebook, dice_score is already in scope

IMG_SIZE       = 128
VOLUME_SLICES  = 100
VOLUME_START   = 22


# ── Helpers ──────────────────────────────────────────────────────────────────

def load_patient_slices(patient_id, data_path, slices=None):
    """
    Load all (or a subset of) 2-channel slices for one patient.
    Returns images (N,2,H,W) and masks (N,H,W) as numpy arrays.
    """
    path = os.path.join(data_path, patient_id)
    flair = nib.load(os.path.join(path, f'{patient_id}_flair.nii')).get_fdata()
    t1ce  = nib.load(os.path.join(path, f'{patient_id}_t1ce.nii')).get_fdata()
    seg   = nib.load(os.path.join(path, f'{patient_id}_seg.nii')).get_fdata()

    slice_indices = slices if slices is not None else range(VOLUME_SLICES)
    images, masks = [], []

    for s in slice_indices:
        sl = s + VOLUME_START
        fl  = cv2.resize(flair[:, :, sl], (IMG_SIZE, IMG_SIZE))
        t1  = cv2.resize(t1ce[:, :, sl],  (IMG_SIZE, IMG_SIZE))
        sg  = cv2.resize(seg[:, :, sl],    (IMG_SIZE, IMG_SIZE),
                         interpolation=cv2.INTER_NEAREST)
        img = np.stack([fl, t1], axis=0)  # (2, H, W)
        mx  = img.max()
        if mx > 0:
            img = img / mx
        sg[sg == 4] = 3                   # remap label 4 → 3
        images.append(img)
        masks.append(sg)

    return np.stack(images), np.stack(masks)   # (N,2,H,W), (N,H,W)


def has_tumor(mask):
    """True if the mask contains at least one non-background pixel."""
    return mask.max() > 0


# ── Core sampler ─────────────────────────────────────────────────────────────

class FewShotSampler:
    """
    Samples (support, query) episode pairs for k-shot evaluation.

    Parameters
    ----------
    support_ids : list[str]   Patient IDs used as the support pool
    query_ids   : list[str]   Patient IDs used as query pool
    data_path   : str
    k_shot      : int         Number of labeled support slices (per class present)
    n_query     : int         Number of query slices per episode
    tumor_only  : bool        If True, only pick slices that contain tumor
    """

    def __init__(self, support_ids, query_ids, data_path,
                 k_shot=5, n_query=16, tumor_only=True):
        self.support_ids = support_ids
        self.query_ids   = query_ids
        self.data_path   = data_path
        self.k_shot      = k_shot
        self.n_query     = n_query
        self.tumor_only  = tumor_only

        # Pre-index slices so episodes are fast to sample
        print("Indexing support patients …")
        self.support_index = self._build_index(support_ids)
        print("Indexing query patients …")
        self.query_index   = self._build_index(query_ids)
        print(f"  Support slices: {len(self.support_index)}")
        print(f"  Query  slices : {len(self.query_index)}")

    # ── private ──────────────────────────────────────────────────

    def _build_index(self, patient_ids):
        """Returns list of (image_2hw, mask_hw) for every valid slice."""
        index = []
        for pid in patient_ids:
            imgs, msks = load_patient_slices(pid, self.data_path)
            for img, msk in zip(imgs, msks):
                if self.tumor_only and not has_tumor(msk):
                    continue
                index.append((img, msk))
        return index

    def _to_tensors(self, items):
        imgs  = torch.tensor(np.stack([x[0] for x in items]), dtype=torch.float32)
        masks = torch.tensor(np.stack([x[1] for x in items]), dtype=torch.long)
        return {'image': imgs, 'mask': masks}

    # ── public ───────────────────────────────────────────────────

    def sample_episode(self):
        """
        Returns
        -------
        support : dict  {'image': (k, 2, H, W), 'mask': (k, H, W)}
        query   : dict  {'image': (n, 2, H, W), 'mask': (n, H, W)}
        """
        support_items = random.sample(self.support_index,
                                      min(self.k_shot, len(self.support_index)))
        query_items   = random.sample(self.query_index,
                                      min(self.n_query,  len(self.query_index)))
        return self._to_tensors(support_items), self._to_tensors(query_items)

    def iter_episodes(self, n_episodes=100):
        """Generator — yields (support, query) tuples."""
        for _ in range(n_episodes):
            yield self.sample_episode()


# ── k-shot fine-tuning baseline ──────────────────────────────────────────────

def kshot_finetune_eval(pretrained_model, sampler, device,
                        lr=1e-4, finetune_steps=10, n_episodes=50):
    """
    Baseline: for each episode, fine-tune a copy of the pretrained model
    on k support slices, then evaluate on query slices.
    Returns list of mean-tumor Dice scores across episodes.
    """
    import copy
    from torch.nn import CrossEntropyLoss

    ce = CrossEntropyLoss()
    episode_dice = []

    for ep, (support, query) in enumerate(sampler.iter_episodes(n_episodes)):
        # Fresh copy per episode so we don't bleed across episodes
        model = copy.deepcopy(pretrained_model).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=lr)

        # ── Fine-tune on support set ──
        model.train()
        s_img  = support['image'].to(device)
        s_mask = support['mask'].to(device)
        for _ in range(finetune_steps):
            opt.zero_grad()
            loss = ce(model(s_img), s_mask)
            loss.backward()
            opt.step()

        # ── Evaluate on query set ──
        model.eval()
        q_img  = query['image'].to(device)
        q_mask = query['mask'].to(device)
        with torch.no_grad():
            preds = torch.argmax(model(q_img), dim=1)

        # Per-class Dice (reuse your existing dice_score fn)
        dice = dice_score(preds, q_mask)   # imported from eval notebook
        episode_dice.append(dice['mean_tumor_dice'])

        if (ep + 1) % 10 == 0:
            print(f"  Episode {ep+1}/{n_episodes} | "
                  f"Mean Dice: {np.mean(episode_dice):.4f}")

    print(f"\nk={sampler.k_shot} shot | "
          f"Mean Dice over {n_episodes} episodes: {np.mean(episode_dice):.4f} "
          f"± {np.std(episode_dice):.4f}")
    return episode_dice


# ── Quick smoke test ──────────────────────────────────────────────────────────
if __name__ == '__main__':
    import kagglehub
    KAGGLE_PATH = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
    TRAIN_DATASET_PATH = None
    for root, dirs, _ in os.walk(KAGGLE_PATH):
        if "MICCAI_BraTS2020_TrainingData" in dirs:
            TRAIN_DATASET_PATH = os.path.join(root, "MICCAI_BraTS2020_TrainingData")
            break
    assert TRAIN_DATASET_PATH, "Could not find MICCAI_BraTS2020_TrainingData"

    from sklearn.model_selection import train_test_split
    all_ids = [f.name for f in os.scandir(TRAIN_DATASET_PATH) if f.is_dir()]
    train_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)

    sampler = FewShotSampler(
        support_ids=train_ids[:10],   # small subset for speed test
        query_ids=val_ids[:5],
        data_path=TRAIN_DATASET_PATH,
        k_shot=5,
        n_query=16,
    )
    support, query = sampler.sample_episode()
    print("Support image shape:", support['image'].shape)  # (5, 2, 128, 128)
    print("Query   mask  shape:", query['mask'].shape)     # (16, 128, 128)